# QGModel — workflow with the 3-phase ensemble recipe

Same pattern as the Lorenz96 notebook, applied to the 1.5-layer quasi-geostrophic model. The state vector is `[q, psi]`, with two natural variable blocks for localization.

## The 3-phase ensemble recipe

TEDA builds the truth and the initial ensemble in three phases. Each phase produces an artifact that can be saved to disk and reused across scenarios — they are the **expensive** parts of an experiment and should not be recomputed every time.

1. **Phase 1 — Reference state `x0_ref`** 
   Start from a synthetic IC and propagate by `spinup_truth`. The result is a state on the model attractor.

2. **Phase 2 — Ensemble centre `xb`** 
   Perturb `x0_ref` with std `pert_xb` and propagate by `spinup_xb`. Chaos amplifies the small initial perturbation, so `xb` ends up meaningfully separated from the trajectory passing through `x0_ref`. This separation represents the **background error** that DA must correct.

3. **Phase 3 — Initial ensemble `X_b`** 
   Perturb `xb` with std `pert_ensemble` once per member and propagate each by `spinup_ensemble`. This disperses the ensemble around `xb` without moving its centre much.

4. **Truth sync** 
   The truth trajectory is propagated from `x0_ref` by `spinup_xb + spinup_ensemble` so it lands at the same model time as the ensemble. The filter then sees a realistic gap between `xb` and `truth[0]`.

Once `x0_ref`, `xb`, and `initial_ensemble` are saved to disk, generating a new scenario with a different observation seed is **essentially instant** — only the observations and operator depend on the scenario seed.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import os

from pyteda.models import QGModel
from pyteda.observation import LinearSelection, IsotropicDiagonal
from pyteda.experiments import Scenario, Benchmark
from pyteda.io import (
    save_state_vector, load_state_vector,
    save_initial_ensemble, load_initial_ensemble,
    get_data_dir,
)

SCEN_DIR = get_data_dir()
print('Storing artifacts in:', SCEN_DIR)

## QG-specific: pluggable integrators and initial conditions

The QG core ships with a registry of time-stepping schemes and a library of initial conditions, exposed through static methods on `QGModel`.

In [ ]:
print('integrators :', QGModel.list_available_integrators())
print('initial cond:', QGModel.list_available_ics())

## 1. Phase 1 — Build `x0_ref`

In [ ]:
model = QGModel(
    mrefin=4, scheme='rk4', dt=0.5, bc='channel',
    ic_kind='fourier', ic_kwargs={'amplitude': 0.5}, verbose=False,
)
n = model.get_number_of_variables()
M_OBS = 80
ENSEMBLE_SIZE = 12
print('state dim:', n, '   var_blocks:', list(model.var_blocks.keys()))

X0REF_PATH = f'{SCEN_DIR}/x0_ref_qg.nc'
if not os.path.exists(X0REF_PATH):
    print('Building x0_ref ...')
    x0 = model.get_initial_condition()
    x0_ref = model.propagate(x0, np.array([0.0, 50.0]))
    save_state_vector(x0_ref, X0REF_PATH, name='x0_ref',
                      meta={'model': 'QGModel', 'spinup_truth': 50.0})
x0_ref = load_state_vector(X0REF_PATH)
print('x0_ref shape:', x0_ref.shape)

## 2. Phase 2 — Build `xb`

In [ ]:
XB_PATH = f'{SCEN_DIR}/xb_qg.nc'
if not os.path.exists(XB_PATH):
    print('Building xb ...')
    rng_xb = np.random.default_rng(seed=999)
    xb_init = x0_ref + 0.5 * rng_xb.standard_normal(n)
    xb = model.propagate(xb_init, np.array([0.0, 30.0]))
    save_state_vector(xb, XB_PATH, name='xb',
                      meta={'pert_xb': 0.5, 'spinup_xb': 30.0, 'seed': 999})
xb = load_state_vector(XB_PATH)
rel_sep = np.linalg.norm(xb - x0_ref) / np.linalg.norm(x0_ref)
print(f'||xb - x0_ref|| / ||x0_ref|| = {rel_sep:.3f}')

## 3. Phase 3 — Build `X_b`

In [ ]:
X0_PATH = f'{SCEN_DIR}/X0_qg.nc'
if not os.path.exists(X0_PATH):
    print('Building initial ensemble ...')
    rng_ens = np.random.default_rng(seed=1000)
    X0 = np.empty((n, ENSEMBLE_SIZE))
    for k in range(ENSEMBLE_SIZE):
        x_member = xb + 0.02 * rng_ens.standard_normal(n)
        X0[:, k] = model.propagate(x_member, np.array([0.0, 2.0]))
    save_initial_ensemble(X0, X0_PATH,
                          meta={'model': 'QGModel', 'pert_ensemble': 0.02,
                                'spinup_ensemble': 2.0, 'ensemble_size': ENSEMBLE_SIZE})
X0 = load_initial_ensemble(X0_PATH)
print('X0 shape:', X0.shape)

## 4. Generate scenarios

In [ ]:
SEEDS = [42, 43, 44]
scenario_paths = [f'{SCEN_DIR}/qg_seed{s}.nc' for s in SEEDS]

for seed, path in zip(SEEDS, scenario_paths):
    if os.path.exists(path):
        continue
    scen = Scenario.generate(
        model=model,
        operator_factory=lambda rng: LinearSelection(m=M_OBS, n_state=n, rng=rng),
        noise=IsotropicDiagonal(std=0.01, dim=M_OBS),
        x0_ref=x0_ref, xb=xb, initial_ensemble=X0,
        spinup_xb=30.0, spinup_ensemble=5.0,
        obs_freq=2.0, end_time=8.0, seed=seed,
    )
    scen.save(path)
    print(f'  seed={seed}  hash={scen.meta["config_hash"]}  saved to {path}')

## 5. Load and run benchmark

Note the **per-block localization radius** for LETKF: `r={'q': 2, 'psi': 4}` lets `q` use a tight localization and `psi` a wider one.

In [ ]:
scenarios = [Scenario.load(p, model=model) for p in scenario_paths]

methods = {
    'EnKF':            dict(method='enkf'),
    'EnKF-MC(r=2)':    dict(method='enkf-modified-cholesky', r=2),
    "LETKF q=2 psi=4": dict(method='letkf', r={'q': 2, 'psi': 4}),
}

results = Benchmark(
    scenarios=scenarios, methods=methods,
    n_runs_per_method=2, inflation_factor=1.04,
    method_seed_base=1000, parallel=False, verbose=False,
    store_diagnostics=True,
).run()
print(f'Total cells: {len(results.rows)}')

## 6. Summary

In [ ]:
results.summary_table()

## 7. Calibration diagnostics

In [ ]:
results.diagnostics_summary()

## 8. Error curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
results.plot_error_curves(ax=ax, kind='analysis')
ax.set_title('QGModel — analysis RMSE (median + IQR)')
plt.tight_layout(); plt.show()

## 9. Spread vs error

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
results.plot_spread_vs_error(ax=ax, kind='analysis')
ax.set_title('QGModel — analysis spread vs RMSE')
plt.tight_layout(); plt.show()

## 10. Rank histogram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
results.plot_rank_histogram('LETKF q=2 psi=4', kind='analysis', ax=ax)
plt.tight_layout(); plt.show()